In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import math

rand_value = 2909  # random seed for reproducibility

# Functions

def euclid_distance_matrix(A, B):
    """Compute pairwise Euclidean distances between points in A and B."""
    A = np.array(A)
    B = np.array(B)
    sum_sq_A = np.sum(A**2, axis=1).reshape(-1,1)
    sum_sq_B = np.sum(B**2, axis=1).reshape(1,-1)
    AB = A.dot(B.T)
    dist_sq = sum_sq_A + sum_sq_B - 2*AB
    dist_sq[dist_sq < 0] = 0.0  
    return np.sqrt(dist_sq)

def predict_1nn(X_train, y_train, X_test):
    """Predict labels using 1-Nearest Neighbour."""
    dist_matrix = euclid_distance_matrix(X_test, X_train)
    nearest_idx = np.argmin(dist_matrix, axis=1)
    predicted = y_train[nearest_idx]
    distances = dist_matrix[np.arange(len(X_test)), nearest_idx]
    return predicted, distances

def conformity_scores(X, y):
    """
    Compute conformity scores for a dataset.
    Conformity = distance to nearest different class / distance to nearest same class
    """
    n_samples = X.shape[0]
    dist = euclid_distance_matrix(X, X)
    np.fill_diagonal(dist, math.inf)

    nearest_same = np.full(n_samples, math.inf)
    nearest_diff = np.full(n_samples, math.inf)

    for i in range(n_samples):
        same_mask = (y == y[i])
        diff_mask = ~same_mask
        same_mask[i] = False  

        if np.any(same_mask):
            nearest_same[i] = np.min(dist[i, same_mask])
        if np.any(diff_mask):
            nearest_diff[i] = np.min(dist[i, diff_mask])

    
    conformity = np.zeros(n_samples)
    for i in range(n_samples):
        ds = nearest_same[i]
        dd = nearest_diff[i]
        if ds == 0:
            if dd > 0:
                conformity[i] = math.inf
            else:
                conformity[i] = 0.0
        else:
            conformity[i] = dd / ds

    return conformity, nearest_same, nearest_diff

def augmented_conformity(X_train, y_train, x_new, y_new):
    """Compute conformity scores after adding a new sample."""
    X_aug = np.vstack([X_train, x_new.reshape(1, -1)])
    y_aug = np.append(y_train, y_new)
    c_aug, _, _ = conformity_scores(X_aug, y_aug)
    return c_aug

def nonconformity(c):
    """Convert conformity score to nonconformity measure."""
    alpha = np.zeros_like(c)
    for idx, val in enumerate(c):
        if val == 0:
            alpha[idx] = math.inf
        elif math.isinf(val):
            alpha[idx] = 0
        else:
            alpha[idx] = 1.0 / val
    return alpha


# Iris dataset

iris = load_iris()
X, y = iris.data, iris.target

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=rand_value, stratify=y
)

# 1-NN predictions
y_pred, _ = predict_1nn(X_train, y_train, X_test)
errors = np.sum(y_pred != y_test)
error_rate = errors / len(y_test)
print(f"Iris 1-NN errors: {errors}/{len(y_test)}, rate = {error_rate:.4f}")

# Conformal predictor
c_train, _, _ = conformity_scores(X_train, y_train)
alpha_train = nonconformity(c_train)

labels = np.unique(y_train)
pvals = np.zeros((len(X_test), len(labels)))

for i, x in enumerate(X_test):
    for j, label in enumerate(labels):
        c_aug = augmented_conformity(X_train, y_train, x, label)
        alpha_aug = nonconformity(c_aug)
        alpha_new = alpha_aug[-1]
        pvals[i, j] = np.sum(alpha_aug >= alpha_new) / len(alpha_aug)

# Average false p-value
avg_false_p = []
for i in range(len(X_test)):
    true_label = y_test[i]
    wrong_idx = np.where(labels != true_label)[0]
    if len(wrong_idx) > 0:
        avg_false_p.append(np.mean(pvals[i, wrong_idx]))
    else:
        avg_false_p.append(0.0)
avg_false_p_iris = np.mean(avg_false_p)
print(f"Iris conformal: average false p-value = {avg_false_p_iris:.6f}")


# Ionosphere dataset

data = np.genfromtxt('ionosphere.txt', delimiter=',', dtype=float)
X_iono = data[:, :-1]
y_iono = data[:, -1]

# Convert labels to integers 0/1
unique_labels = np.unique(y_iono)
label_map = {lab: idx for idx, lab in enumerate(unique_labels)}
y_mapped = np.array([label_map[lab] for lab in y_iono])

# Split dataset
X_train_iono, X_test_iono, y_train_iono, y_test_iono = train_test_split(
    X_iono, y_mapped, test_size=0.25, random_state=rand_value, stratify=y_mapped
)

# 1-NN predictions
y_pred_iono, _ = predict_1nn(X_train_iono, y_train_iono, X_test_iono)
errors_iono = np.sum(y_pred_iono != y_test_iono)
error_rate_iono = errors_iono / len(y_test_iono)
print(f"Ionosphere 1-NN errors: {errors_iono}/{len(y_test_iono)}, rate = {error_rate_iono:.4f}")

# Conformal predictor
c_train_iono, _, _ = conformity_scores(X_train_iono, y_train_iono)
alpha_train_iono = nonconformity(c_train_iono)

labels_iono = np.unique(y_train_iono)
pvals_iono = np.zeros((len(X_test_iono), len(labels_iono)))

for i, x in enumerate(X_test_iono):
    for j, label in enumerate(labels_iono):
        c_aug = augmented_conformity(X_train_iono, y_train_iono, x, label)
        alpha_aug = nonconformity(c_aug)
        alpha_new = alpha_aug[-1]
        pvals_iono[i, j] = np.sum(alpha_aug >= alpha_new) / len(alpha_aug)

# Average false p-value
avg_false_p_iono = []
for i in range(len(X_test_iono)):
    true_label = y_test_iono[i]
    wrong_idx = np.where(labels_iono != true_label)[0]
    if len(wrong_idx) > 0:
        avg_false_p_iono.append(np.mean(pvals_iono[i, wrong_idx]))
    else:
        avg_false_p_iono.append(0.0)
avg_false_p_iono = np.mean(avg_false_p_iono)
print(f"Ionosphere conformal: average false p-value = {avg_false_p_iono:.6f}")


# Summary of results

print("\n--- REQUIRED NUMBERS ---")
print(f"Iris test error rate (1-NN): {error_rate:.6f}")
print(f"Ionosphere test error rate (1-NN): {error_rate_iono:.6f}")
print(f"Iris average false p-value: {avg_false_p_iris:.6f}")
print(f"Ionosphere average false p-value: {avg_false_p_iono:.6f}")

results = {
    'iris_error_rate': error_rate,
    'iono_error_rate': error_rate_iono,
    'iris_avg_false_p': avg_false_p_iris,
    'iono_avg_false_p': avg_false_p_iono,
    'iris_p_values': pvals,
    'iono_p_values': pvals_iono,
    'iris_pred': y_pred,
    'iono_pred': y_pred_iono,
    'iris_test_true': y_test,
    'iono_test_true': y_test_iono
}

results

Iris 1-NN errors: 3/38, rate = 0.0789
Iris conformal: average false p-value = 0.011295
Ionosphere 1-NN errors: 16/88, rate = 0.1818
Ionosphere conformal: average false p-value = 0.064997

--- REQUIRED NUMBERS ---
Iris test error rate (1-NN): 0.078947
Ionosphere test error rate (1-NN): 0.181818
Iris average false p-value: 0.011295
Ionosphere average false p-value: 0.064997


{'iris_error_rate': 0.07894736842105263,
 'iono_error_rate': 0.18181818181818182,
 'iris_avg_false_p': 0.01129482999534234,
 'iono_avg_false_p': 0.0649965564738292,
 'iris_p_values': array([[0.00884956, 0.00884956, 0.67256637],
        [0.97345133, 0.00884956, 0.00884956],
        [0.00884956, 0.39823009, 0.00884956],
        [0.00884956, 0.00884956, 0.59292035],
        [0.7079646 , 0.00884956, 0.00884956],
        [0.01769912, 0.01769912, 1.        ],
        [0.00884956, 0.00884956, 0.43362832],
        [0.00884956, 0.00884956, 0.44247788],
        [0.00884956, 0.28318584, 0.00884956],
        [0.00884956, 0.83185841, 0.00884956],
        [0.6460177 , 0.00884956, 0.00884956],
        [0.00884956, 0.04424779, 0.04424779],
        [0.00884956, 0.47787611, 0.00884956],
        [0.72566372, 0.00884956, 0.00884956],
        [0.78761062, 0.00884956, 0.00884956],
        [0.00884956, 0.01769912, 0.09734513],
        [0.00884956, 0.52212389, 0.00884956],
        [0.00884956, 0.5840708 , 0.0